# Soluciones — Clase 26: Procesamiento de Texto — NLP Básico

> Este notebook contiene las soluciones completas a todos los ejercicios.
> Intenta resolverlos por tu cuenta antes de consultar aquí.

In [ ]:
# Imports comunes para todas las soluciones
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

In [ ]:
# Solución Ejercicio 1 — Preprocesamiento manual
texto = '¡¡¡El envío llegó MUY TARDE!!! Me prometieron 3 días y tardó 15. Además, el producto estaba dañado. NO lo recomiendo para nada.'

# Paso 1: minúsculas
texto_lower = texto.lower()
print('Paso 1 (minúsculas):', texto_lower)

# Paso 2: eliminar puntuación
texto_limpio = re.sub(r'[^\w\s]', ' ', texto_lower)
texto_limpio = re.sub(r'\d+', '', texto_limpio)  # eliminar números
texto_limpio = re.sub(r'\s+', ' ', texto_limpio).strip()  # espacios extra
print('Paso 2 (sin puntuación):', texto_limpio)

# Paso 3: eliminar stop words
stop_words = {'el', 'la', 'me', 'lo', 'y', 'no', 'además', 'de', 'a', 'para', 'en', 'que', 'se'}
palabras = texto_limpio.split()
texto_final = ' '.join([p for p in palabras if p not in stop_words and len(p) > 2])
print('Paso 3 (sin stop words):', texto_final)

In [ ]:
# Solución Ejercicio 2 — CountVectorizer con frases simples
frases = [
    'el producto es excelente muy recomendado',
    'muy malo no lo recomiendo para nada',
    'llegó rápido producto bien empacado',
    'producto defectuoso muy decepcionante',
    'calidad excelente precio justo llegó rápido'
]

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(frases)

print(f'Vocabulario ({len(vectorizer.vocabulary_)} palabras):')
print(sorted(vectorizer.vocabulary_.keys()))

df_bow = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
print('\nMatriz Bag of Words:')
print(df_bow.to_string())

# Sin stop words predefinidas
vectorizer_clean = CountVectorizer(stop_words=['el', 'la', 'lo', 'muy', 'no', 'para'])
X_clean = vectorizer_clean.fit_transform(frases)
print(f'\nVocabulario sin stop words: {len(vectorizer_clean.vocabulary_)} palabras')
print('Vocabulario limpio:', sorted(vectorizer_clean.vocabulary_.keys()))

In [ ]:
# Solución Ejercicio 3 — TF-IDF
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(frases)

df_tfidf = pd.DataFrame(X_tfidf.toarray(), columns=tfidf.get_feature_names_out()).round(3)
print('Matriz TF-IDF:')
print(df_tfidf.to_string())

print('\nDiferencia clave con BoW:')
print('- "producto" aparece en frases 1,3,4 → IDF bajo → TF-IDF más bajo')
print('- "decepcionante" solo aparece en frase 4 → IDF alto → TF-IDF alto')
print('\nPalabra con mayor TF-IDF en frase 1 (excelente):')
frase1_scores = df_tfidf.iloc[0].sort_values(ascending=False)
print(frase1_scores[frase1_scores > 0].head(3))

In [ ]:
# Solución Ejercicio 5 — Función de preprocesamiento
stop_words_es = {
    'el', 'la', 'los', 'las', 'un', 'una', 'de', 'del', 'al', 'en',
    'que', 'y', 'a', 'se', 'no', 'me', 'te', 'lo', 'le', 'su', 'por',
    'con', 'para', 'es', 'son', 'fue', 'era', 'pero', 'si', 'como',
    'mi', 'tu', 'ya', 'o', 'e', 'ni', 'hay', 'he', 'ha', 'han',
    'muy', 'mas', 'tan', 'bien', 'mal', 'estar', 'ser'
}

def preprocesar_texto(texto):
    """Pipeline completo de preprocesamiento de texto."""
    texto = str(texto).lower()
    texto = re.sub(r'[^\w\s]', ' ', texto)
    texto = re.sub(r'\d+', '', texto)
    palabras = texto.split()
    palabras = [p for p in palabras if p not in stop_words_es and len(p) > 2]
    return ' '.join(palabras)

# Verificar con ejemplo
ejemplo = '¡Llegó muy rápido!! El producto es excelente, lo recomiendo al 100%.'
print('Original:', ejemplo)
print('Procesado:', preprocesar_texto(ejemplo))

# Cargar y aplicar al dataset
df = pd.read_csv('../../datasets/comentarios_productos.csv')
df['texto_limpio'] = df['comentario'].apply(preprocesar_texto)
print('\nDistribución de sentimientos:')
print(df['sentimiento'].value_counts())

In [ ]:
# Solución Ejercicio 6 — Clasificador completo
X_raw = df['texto_limpio']
y = df['sentimiento']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

# TF-IDF con bigramas
tfidf = TfidfVectorizer(max_features=500, ngram_range=(1, 2))
X_train = tfidf.fit_transform(X_train_raw)
X_test = tfidf.transform(X_test_raw)  # Solo transform en test

modelo = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
modelo.fit(X_train, y_train)

y_pred = modelo.predict(X_test)
print(classification_report(y_test, y_pred))

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred, labels=modelo.classes_)
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=modelo.classes_, yticklabels=modelo.classes_, cmap='Blues')
plt.title('Matriz de confusión')
plt.ylabel('Real')
plt.xlabel('Predicho')
plt.tight_layout()
plt.show()

print('\nNota sobre fit vs fit_transform:')
print('En train: fit_transform aprende el vocabulario Y transforma los textos.')
print('En test: solo transform aplica el vocabulario ya aprendido.')
print('Si usáramos fit_transform en test, el vocabulario sería diferente = data leakage.')

In [ ]:
# Solución Ejercicio 7 — Palabras más importantes
feature_names = tfidf.get_feature_names_out()
clases = modelo.classes_
colores = ['#2ecc71', '#e74c3c', '#3498db', '#f39c12']

fig, axes = plt.subplots(1, len(clases), figsize=(5 * len(clases), 5))

for idx, clase in enumerate(clases):
    coefs = modelo.coef_[idx]
    top_indices = coefs.argsort()[-12:][::-1]
    top_palabras = feature_names[top_indices]
    top_scores = coefs[top_indices]

    axes[idx].barh(top_palabras[::-1], top_scores[::-1],
                   color=colores[idx % len(colores)], edgecolor='white')
    axes[idx].set_title(f'{clase}', fontsize=11)
    axes[idx].set_xlabel('Coeficiente')
    axes[idx].grid(axis='x', alpha=0.3)

plt.suptitle('Palabras más importantes por sentimiento', fontsize=13)
plt.tight_layout()
plt.show()

print('Con ngram_range=(1,2) aparecen bigramas como "no recomiendo" o "muy bueno".')
print('Esto mejora la detección de negaciones y frases compuestas.')

In [ ]:
# Solución Ejercicio 8 — Predicción interactiva
def predecir_sentimiento(texto):
    texto_limpio = preprocesar_texto(texto)
    X_nuevo = tfidf.transform([texto_limpio])
    prediccion = modelo.predict(X_nuevo)[0]
    probabilidades = modelo.predict_proba(X_nuevo)[0]
    
    print(f'Comentario: "{texto}"')
    print(f'Sentimiento predicho: {prediccion}')
    for clase, prob in sorted(zip(modelo.classes_, probabilidades), key=lambda x: -x[1]):
        barra = '█' * int(prob * 25)
        print(f'  {clase:12s}: {prob:.1%}  {barra}')
    print()

predecir_sentimiento('Llegó perfectamente y funciona de maravilla, muy satisfecho con la compra')
predecir_sentimiento('Una basura total, se rompió a los dos días y el servicio no responde')
predecir_sentimiento('El producto está bien, nada especial, cumple su función')
print('El modelo usa las palabras aprendidas durante el entrenamiento para clasificar.')
print('Palabras desconocidas (fuera del vocabulario) son ignoradas.')